In [1]:
# Cell 1: Imports and Setup
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Libraries imported successfully")

Libraries imported successfully


C:\Users\himan\AppData\Local\Temp\ipykernel_18152\2290079924.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
# Cell 2: Load Raw Data (Bronze Layer)
raw_path = Path("../data/raw/zomato_delivery_raw.csv")
df = pd.read_csv(raw_path)

print(f"Loaded raw data: {df.shape[0]:,} rows, {df.shape[1]} columns")
print("Preserving original raw data untouched.")

Loaded raw data: 38,964 rows, 22 columns
Preserving original raw data untouched.


In [3]:
# Cell 3: Standardize Column Names
# Convert to lowercase and replace spaces/special chars with underscores
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')

# Verify changes
print("Updated columns:")
print(df.columns.tolist())

Updated columns:
['id', 'delivery_person_id', 'delivery_person_age', 'delivery_person_ratings', 'restaurant_latitude', 'restaurant_longitude', 'delivery_location_latitude', 'delivery_location_longitude', 'order_date', 'time_orderd', 'time_order_picked', 'weather_conditions', 'road_traffic_density', 'vehicle_condition', 'type_of_order', 'type_of_vehicle', 'multiple_deliveries', 'festival', 'city', 'time_taken_min', 'distance_km', 'delivery_speed']


In [4]:
# Cell 4: Fix Date Parsing (Crucial Fix)
# The dates are DD-MM-YYYY. We must specify dayfirst=True.
df['order_date'] = pd.to_datetime(df['order_date'], dayfirst=True, errors='coerce')

# Extract time-based features
df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['day_of_week'] = df['order_date'].dt.day_name()
df['day_of_month'] = df['order_date'].dt.day

print(f"Date parsing complete. Valid dates: {df['order_date'].notna().sum():,}")
print(f"Date range: {df['order_date'].min()} to {df['order_date'].max()}")

Date parsing complete. Valid dates: 38,964
Date range: 2022-02-11 00:00:00 to 2022-04-06 00:00:00


In [5]:
# Cell 5: Handle Missing Values (Conservative Approach)
# We will NOT drop rows. We will impute numeric values and flag missing times.

# 1. Impute Age and Ratings with Median (most robust to outliers)
age_median = df['delivery_person_age'].median()
rating_median = df['delivery_person_ratings'].median()

df['delivery_person_age'] = df['delivery_person_age'].fillna(age_median)
df['delivery_person_ratings'] = df['delivery_person_ratings'].fillna(rating_median)

print(f"Imputed Age with median: {age_median}")
print(f"Imputed Ratings with median: {rating_median}")

# 2. Handle missing Time_Orderd
# We create a flag to track if the order time was missing
df['time_orderd_missing'] = df['time_orderd'].isna()

# 3. Parse Time_Order_picked to calculate pickup delay later
# Since Time_Orderd is missing for some, we will handle pickup delay carefully in the next cell.

Imputed Age with median: 30.0
Imputed Ratings with median: 4.7


In [6]:
# Cell 6: Fix Categorical Typos and Standardize
# Fix the "Metropolitian" typo
df['city'] = df['city'].str.strip().replace('Metropolitian', 'Metropolitan')

# Standardize other categorical columns (strip whitespace)
cat_cols = ['weather_conditions', 'road_traffic_density', 'type_of_order', 'type_of_vehicle', 'festival', 'delivery_speed']
for col in cat_cols:
    df[col] = df[col].str.strip().str.title() # e.g., "fog" -> "Fog", "jam" -> "Jam"

print("Categorical columns standardized.")
print("City unique values:", df['city'].unique())

Categorical columns standardized.
City unique values: ['Metropolitan' 'Urban' 'Semi-Urban']


In [7]:
# Cell 7: Feature Engineering (Gold Layer)
# 1. Parse Time_Orderd and Time_Order_picked to calculate pickup delay
def parse_time(time_str):
    """Safely parse HH:MM string to datetime time object."""
    if pd.isna(time_str):
        return np.nan
    try:
        return pd.to_datetime(str(time_str), format='%H:%M').time()
    except:
        return np.nan

df['time_orderd_parsed'] = df['time_orderd'].apply(parse_time)
df['time_order_picked_parsed'] = df['time_order_picked'].apply(parse_time)

# Calculate pickup delay in minutes
def calc_pickup_delay(row):
    if pd.isna(row['time_orderd_parsed']) or pd.isna(row['time_order_picked_parsed']):
        return np.nan
    
    # Convert to datetime for subtraction (using a dummy date)
    t1 = pd.to_datetime(f"2000-01-01 {row['time_orderd_parsed']}")
    t2 = pd.to_datetime(f"2000-01-01 {row['time_order_picked_parsed']}")
    
    diff = (t2 - t1).total_seconds() / 60
    # Handle midnight crossover (if picked up next day, though unlikely for food delivery)
    if diff < 0:
        diff += 24 * 60
    return diff

df['pickup_delay_minutes'] = df.apply(calc_pickup_delay, axis=1)

# 2. Extract Order Hour
df['order_hour'] = df['time_orderd_parsed'].apply(lambda x: x.hour if pd.notna(x) else np.nan)

# 3. Create Time of Day category
def get_time_of_day(hour):
    if pd.isna(hour):
        return 'Unknown'
    if 6 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

df['time_of_day'] = df['order_hour'].apply(get_time_of_day)

print("Feature engineering complete.")
print(f"Pickup delay calculated for {df['pickup_delay_minutes'].notna().sum():,} rows.")

Feature engineering complete.
Pickup delay calculated for 30,814 rows.


In [8]:
# Cell 8: Save Cleaned Data (Silver/Gold Layer)
processed_dir = Path("../data/processed")
processed_dir.mkdir(exist_ok=True)

cleaned_path = processed_dir / "zomato_cleaned_analytical.csv"

# Drop helper columns used during cleaning
df_final = df.drop(columns=['time_orderd_parsed', 'time_order_picked_parsed'])

df_final.to_csv(cleaned_path, index=False)

print("=" * 60)
print("DATA CLEANING COMPLETE")
print("=" * 60)
print(f"Saved to: {cleaned_path}")
print(f"Final shape: {df_final.shape[0]:,} rows × {df_final.shape[1]} columns")
print("\nData is now ready for Analytics and Power BI.")

DATA CLEANING COMPLETE
Saved to: ..\data\processed\zomato_cleaned_analytical.csv
Final shape: 38,964 rows × 30 columns

Data is now ready for Analytics and Power BI.
